<a href="https://colab.research.google.com/github/Hema-14052005/Hema-14052005/blob/main/food_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn fuzzywuzzy python-Levenshtein
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from fuzzywuzzy import fuzz, process

# Load the dataset
food_dataset = pd.read_excel("/content/final food db.xlsx")

# Define features and target variables
features = ['Food_items', 'Breakfast', 'Lunch', 'Dinner', 'VegNovVeg']
target = ['Calories', 'Fats', 'Proteins', 'Iron', 'Calcium', 'Carbohydrates', 'Fibre', 'VitaminD']
target_recommend = 'Food_Style_Comment_and_Recommendation'

# Preprocessing
X = food_dataset[features].copy()
y = food_dataset[target]
y_recommend = food_dataset[target_recommend]

# Keep original case for Food_items during label encoding
label_encoder_food = LabelEncoder()
X.loc[:, 'Food_items'] = label_encoder_food.fit_transform(X['Food_items'])

label_encoder_veg = LabelEncoder()
all_veg_values = pd.concat([food_dataset['VegNovVeg'].astype(str), pd.Series(['Veg', 'veg', 'Non-veg', 'non-veg'])], ignore_index=True)
label_encoder_veg.fit(all_veg_values)
X.loc[:, 'VegNovVeg'] = label_encoder_veg.transform(X['VegNovVeg'].astype(str))

# Fit StandardScaler
sc = StandardScaler()
x = sc.fit_transform(X[['Food_items', 'VegNovVeg']])

# Split data
X_train, X_test, y_train, y_test, y_train_recommend, y_test_recommend = train_test_split(
    x, y, y_recommend, test_size=0.2, random_state=2
)

# Train the model
model = RandomForestRegressor(random_state=2)
model.fit(X_train, y_train)

# Create TF-IDF matrix (if still needed)
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(y_recommend)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to get recommendations (with case sensitivity and fuzzy matching)
def get_recommendations(food_items_with_quantities, top_n=5):
    total_nutrient_predictions = np.zeros(len(target))
    food_style_comments = []

    for food_item_with_quantity in food_items_with_quantities:
        match = re.match(r"([a-zA-Z\s]+)(\d+)?", food_item_with_quantity)
        if match:
            food_item = match.group(1).strip()  # Keep original case
            quantity = int(match.group(2)) if match.group(2) else 1
        else:
            food_item = food_item_with_quantity
            quantity = 1

        # Fuzzy Matching (using process.extractOne for case-insensitive matching)
        closest_match = process.extractOne(food_item, food_dataset['Food_items'], scorer=fuzz.token_sort_ratio)

        if closest_match and closest_match[1] >= 60:  # Adjust threshold as needed
            food_item = closest_match[0]  # Use the closest match
        else:
            print(f"Warning: Food item '{food_item}' not found in the dataset. Skipping...")
            continue

        # Create input features for prediction (excluding breakfast, lunch, dinner)
        input_data = {
            'Food_items': food_item,  # Use original case
            'VegNovVeg': 'Veg' if food_item in food_dataset[food_dataset['VegNovVeg'].isin(['Veg', 'veg'])]['Food_items'].values else 'Non-veg'  # Use original case
        }

        input_features = pd.DataFrame([input_data], columns=['Food_items', 'VegNovVeg'])

        try:
            input_features.loc[:, 'Food_items'] = label_encoder_food.transform(input_features['Food_items'])
        except ValueError as e:
            print(f"Warning: Food item '{food_item}' not found in the dataset. Skipping...")
            continue

        input_features.loc[:, 'VegNovVeg'] = label_encoder_veg.transform(input_features['VegNovVeg'].astype(str))
        input_features_scaled = sc.transform(input_features[['Food_items', 'VegNovVeg']])
        nutrient_predictions = model.predict(input_features_scaled)[0]
        nutrient_predictions = nutrient_predictions * quantity
        total_nutrient_predictions += nutrient_predictions

        if food
            food_index = label_encoder_food.transform([food_item])[0]
            food_style_comment = food_dataset.loc[food_index, 'Food_Style_Comment_and_Recommendation']
            food_style_comments.extend([food_style_comment] * quantity)
        else:
            print(f"Warning: Food item '{food_item}' not found in the dataset for recommendation. Skipping...")
            continue

    predicted_nutrients = dict(zip(target, total_nutrient_predictions))

    # Print nutrient information
    print("Nutritional values:")
    for nutrient in target:
        print(f"{nutrient}: {predicted_nutrients[nutrient]:.2f}")
    print("________________________________________________________________________")
   # if target[calories]>=1200
    print("today's food review")

    # Print Food_Style_Comment_and_Recommendation
    print("\nFood Recommendation:")
    if food_style_comments:
        most_common_comment = Counter(food_style_comments).most_common(1)[0][0]
        print(most_common_comment)
    else:
        print("No recommendations found for the provided food items.")

# Get user input
food_items_str = input("Enter food items with quantities (e.g., idly 2, dosa 1, vada 3): ")
food_items_with_quantities = [item.strip() for item in food_items_str.split(',')]
food_item_match = re.match(r"([\w\s-]+)\s*(\d+)?", food_item_with_quantity)

# Get recommendations
get_recommendations(food_item_match)

SyntaxError: expected ':' (<ipython-input-1-e48806822174>, line 96)

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from fuzzywuzzy import fuzz, process
from collections import Counter

# Load the dataset
food_dataset = pd.read_excel("/content/final food db.xlsx")

# Define features and target variables
features = ['Food_items', 'Breakfast', 'Lunch', 'Dinner', 'VegNovVeg']
target = ['Calories', 'Fats', 'Proteins', 'Iron', 'Calcium', 'Carbohydrates', 'Fibre', 'VitaminD']
target_recommend = 'Food_Style_Comment_and_Recommendation'

# Preprocessing
X = food_dataset[features].copy()
y = food_dataset[target]
y_recommend = food_dataset[target_recommend]

# Label Encoding for categorical features
label_encoder_food = LabelEncoder()
X['Food_items'] = label_encoder_food.fit_transform(X['Food_items'])

label_encoder_veg = LabelEncoder()
all_veg_values = pd.concat([food_dataset['VegNovVeg'].astype(str), pd.Series(['Veg', 'veg', 'Non-veg', 'non-veg'])], ignore_index=True)
label_encoder_veg.fit(all_veg_values)
X['VegNovVeg'] = label_encoder_veg.transform(X['VegNovVeg'].astype(str))

# Normalize the features
sc = StandardScaler()
X_scaled = sc.fit_transform(X[['Food_items', 'VegNovVeg']])

# Normalize the nutrient target variables
scaler_y = MinMaxScaler()
y_normalized = scaler_y.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test, y_train_recommend, y_test_recommend = train_test_split(
    X_scaled, y_normalized, y_recommend, test_size=0.2, random_state=2
)

# Train the model
model = RandomForestRegressor(random_state=2)
model.fit(X_train, y_train)

# Function to get recommendations with normalization

def get_recommendations(food_items_with_quantities, top_n=5):
    total_nutrient_predictions = np.zeros(len(target))
    food_style_comments = []

    for food_item_with_quantity in food_items_with_quantities:
        match = re.match(r"([\w\s-]+)\s*(\d+)?", food_item_with_quantity)
        if match:
            food_item = match.group(1).strip()
            quantity = int(match.group(2)) if match.group(2) else 1
        else:
            food_item = food_item_with_quantity
            quantity = 1

        # Fuzzy Matching
        closest_match = process.extractOne(food_item, food_dataset['Food_items'], scorer=fuzz.WRatio)
        if closest_match and closest_match[1] >= 80:
            food_item = closest_match[0]
        else:
            print(f"Warning: Food item '{food_item}' not found. Skipping...")
            continue

        # Create input features for prediction
        input_data = {
            'Food_items': food_item,
            'VegNovVeg': 'Veg' if food_item in food_dataset[food_dataset['VegNovVeg'].isin(['Veg', 'veg'])]['Food_items'].values else 'Non-veg'
        }

        input_features = pd.DataFrame([input_data], columns=['Food_items', 'VegNovVeg'])

        try:
            input_features['Food_items'] = label_encoder_food.transform(input_features['Food_items'])
        except ValueError:
            print(f"Warning: Food item '{food_item}' not found in dataset. Skipping...")
            continue

        input_features['VegNovVeg'] = label_encoder_veg.transform(input_features['VegNovVeg'].astype(str))
        input_features_scaled = sc.transform(input_features[['Food_items', 'VegNovVeg']])

        # Predict normalized nutrients and convert back
        nutrient_predictions_normalized = model.predict(input_features_scaled)[0]
        nutrient_predictions = scaler_y.inverse_transform([nutrient_predictions_normalized])[0]  # Convert back
        nutrient_predictions *= quantity  # Adjust for quantity
        total_nutrient_predictions += nutrient_predictions

        # Get recommendations
        try:
            food_index = label_encoder_food.transform([food_item])[0]
            food_style_comment = food_dataset.loc[food_index, 'Food_Style_Comment_and_Recommendation']
            food_style_comments.extend([food_style_comment] * quantity)
        except ValueError:
            continue

    predicted_nutrients = dict(zip(target, total_nutrient_predictions))

    # Print nutrient information
    print("Nutritional values:")
    for nutrient, value in predicted_nutrients.items():
        print(f"{nutrient}: {value:.2f}")
    print("________________________________________________________________________")
    print("Today's food review")

    # Print Food_Style_Comment_and_Recommendation
    print("\nFood Recommendation:")
    if food_style_comments:
        most_common_comment = Counter(food_style_comments).most_common(1)[0][0]
        print(most_common_comment)
    else:
        print("No recommendations found for the provided food items.")

# Get user input
food_items_str = input("Enter food items with quantities (e.g., idly 2, dosa 1, vada 3): ")
food_items_with_quantities = [item.strip() for item in food_items_str.split(',')]

# Get recommendations
get_recommendations(food_items_with_quantities)
